# USDA WASDE (Hard Fundamentals)

## How WASDE Impacts Cotton Prices
In agricultural economics, the USDA WASDE report is the definitive "ground truth." While daily futures prices are driven by speculation and momentum, WASDE provides the actual, physical math: Ending Stocks (how much cotton is left) and Total Consumption (how much is being used).

Commodity markets price in expected supply and demand. If a new WASDE report is released and the estimates deviate from what the market expected—for example, signaling that the global supply buffer is much tighter than anticipated—it creates an immediate "step function" in the data. This triggers massive, violent repricing as institutional mills panic-buy to secure inventory, driving prices parabolic.

Because the USDA does not have a simple Yahoo Finance-style library, extracting this data programmatically usually involves either the USDA NASS Quick Stats API (which requires a free API key) or downloading the official Production, Supply, and Distribution (PSD) historical datasets.

This script sets up the extraction pipeline using the requests library. It targets the foundational variables you outlined: Ending Stocks and Total Consumption.

In [12]:
import pandas as pd
import os
import glob

# 1. Directory Setup
base_dir = r"D:\MS_Data_Science_Thesis\Data_Extraction"
wasde_raw_folder = os.path.join(base_dir, "Downloaded_datasets", "WASDE_Folder")
output_folder = os.path.join(base_dir, "Raw_Data_Folder")

def extract_and_combine_wasde():
    print("--- Starting USDA WASDE Bulk Extraction ---")
    
    if not os.path.exists(wasde_raw_folder):
        print(f"Error: Please create the folder {wasde_raw_folder} and put all WASDE CSVs in it.")
        return

    # 2. Find all CSV files in the directory
    all_files = glob.glob(os.path.join(wasde_raw_folder, "*.csv"))
    
    if len(all_files) == 0:
        print("Error: No CSV files found in the WASDE folder.")
        return
        
    print(f"Found {len(all_files)} CSV files. Combining...")

    # 3. Read and concatenate all CSVs
    df_list = []
    for file in all_files:
        try:
            # Some older USDA files might have different encodings, 
            # so we use a broad exception catch just in case.
            temp_df = pd.read_csv(file, low_memory=False)
            df_list.append(temp_df)
        except Exception as e:
            print(f"Warning: Could not read {file}. Error: {e}")
            
    master_df = pd.concat(df_list, ignore_index=True)
    
    # 4. Filter for Cotton
    # Note: Column names might vary slightly across years (e.g., 'Commodity' vs 'commodity').
    # Standardizing column names to title case to be safe.
    master_df.columns = master_df.columns.str.title()
    
    if 'Commodity' not in master_df.columns:
         print("Error: 'Commodity' column not found in combined data.")
         return
         
    cotton_df = master_df[master_df['Commodity'].astype(str).str.contains('Cotton', case=False, na=False)].copy()
    
    # 5. Filter for Target Attributes and Regions
    target_attributes = ['Ending Stocks', 'Total Use', 'Domestic Use', 'Exports']
    target_regions = ['United States', 'World']
    
    cotton_df = cotton_df[cotton_df['Attribute'].isin(target_attributes)]
    cotton_df = cotton_df[cotton_df['Region'].isin(target_regions)]
    
    # 6. Clean and Export
    columns_to_keep = ['Reportdate', 'Region', 'Attribute', 'Value']
    # Handle slight column naming variations
    actual_cols = [col for col in cotton_df.columns if col.lower() in [c.lower() for c in columns_to_keep]]
    
    final_df = cotton_df[actual_cols].copy()
    
    # Rename 'Reportdate' back to standard camelCase if it got altered
    final_df = final_df.rename(columns={col: 'ReportDate' for col in final_df.columns if col.lower() == 'reportdate'})
    
    # Sort chronologically
    final_df['ReportDate'] = pd.to_datetime(final_df['ReportDate'])
    final_df = final_df.sort_values(by='ReportDate')
    
    output_path = os.path.join(output_folder, "wasde_cotton_filtered.csv")
    final_df.to_csv(output_path, index=False)
    
    print(f"Success! Combined and filtered WASDE Cotton data exported to: {output_path}")
    print(f"Data Shape: {final_df.shape}")

if __name__ == "__main__":
    extract_and_combine_wasde()

--- Starting USDA WASDE Bulk Extraction ---
Found 64 CSV files. Combining...


C:\Users\siddh\AppData\Local\Temp\ipykernel_19892\1150551055.py:68: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  final_df['ReportDate'] = pd.to_datetime(final_df['ReportDate'])


Success! Combined and filtered WASDE Cotton data exported to: D:\MS_Data_Science_Thesis\Data_Extraction\Raw_Data_Folder\wasde_cotton_filtered.csv
Data Shape: (14637, 4)
